# PEP 8 Demo — **BEFORE** (Typical Student Code)

This notebook fetches Pokémon data from the free
[PokéAPI](https://pokeapi.co/) and analyzes base stats.

The code **runs correctly** but is full of style violations.
Open **`pep8_after.ipynb`** side-by-side to see every fix.

## Setup

In [ ]:
import requests,json #http + parsing
import pandas as pd,numpy as np #data
import matplotlib.pyplot as plt #viz
from collections import Counter,defaultdict

## Configuration

In [ ]:
baseUrl='https://pokeapi.co/api/v2/' #api root
numPokemon=50 #how many to fetch
statNames=['hp','attack','defense','special-attack','special-defense','speed'] #the 6 base stats

## Fetching data from the API

We call the `/pokemon/{id}` endpoint for each Pokémon.
The API returns JSON with stats, types, height, weight, etc.

In [ ]:
def fetchPokemon(pokemon_id):
    ##
    ## Fetch one pokemon by ID.
    ## Returns a dict with name, types, and base stats,
    ## or None if the request fails.
    ##
    url=baseUrl+f'pokemon/{pokemon_id}' #build url
    response=requests.get(url) #make GET request
    if response.status_code!=200: #check for errors
        print(f'Error fetching id {pokemon_id}: {response.status_code}') #log it
        return None #give up
    data=response.json() #parse JSON body
    #extract the fields we care about
    stats={s['stat']['name']:s['base_stat'] for s in data['stats']} #dict comprehension
    types=[t['type']['name'] for t in data['types']] #list comprehension
    return{'name':data['name'],'id':data['id'],'types':types,'height':data['height'],'weight':data['weight'],**stats} #merge stats in

In [ ]:
#fetch all pokemon
allPokemon=[]
for i in range(1,numPokemon+1):
    result=fetchPokemon(i)
    if result!=None: #should use 'is not None'
        allPokemon.append(result)
    if i%10==0: #progress update every 10
        print(f'Fetched {i}/{numPokemon}...')
print(f'Done! Got {len(allPokemon)} pokemon.')

## Build a DataFrame

Convert the list of dicts to a pandas DataFrame for easier analysis.

In [ ]:
df=pd.DataFrame(allPokemon) #list of dicts -> dataframe
df['total']=df[statNames].sum(axis=1) #compute base stat total
df['primary_type']=df['types'].apply(lambda t:t[0]) #first type is primary
df['height_m']=df['height']/10 #decimeters to meters
df['weight_kg']=df['weight']/10 #hectograms to kilograms
print(f'DataFrame shape: {df.shape}') #rows, cols
df.head(10)

## Analysis class

A reusable class that computes summary statistics
for a given type of Pokémon.

In [ ]:
class pokemonAnalyzer:
    ##
    ## Analyzes a subset of Pokemon filtered by type.
    ## Provides methods for stat summaries and rankings.
    ##
    def __init__(self,dataframe,type_filter=None):
        self.typeName=type_filter #which type we're looking at
        if type_filter==None: #should be 'is None'
            self.data=dataframe.copy() #use all data
        else:
            self.data=dataframe[dataframe['primary_type']==type_filter].copy() #filter by type
        self.count=len(self.data) #how many pokemon
    def statSummary(self):
        ##
        ## Return mean of each base stat.
        ##
        if self.count==0:
            return None
        return self.data[statNames].mean().round(1).to_dict() #mean across all pokemon
    def topN(self,stat='total',n=5):
        ##
        ## Return the top N pokemon by a given stat.
        ##
        return self.data.nlargest(n,stat)[['name',stat]].reset_index(drop=True) #sorted descending
    def printReport(self):
        label=self.typeName if self.typeName else 'All Types' #display label
        print(f'--- {label} ({self.count} pokemon) ---')
        stats=self.statSummary()
        if stats==None: #should be 'is None'
            print('  No data');return #semicolon bad
        for statName,val in stats.items(): #iterate stats
            print(f'  {statName}: {val}') #print each

## Run the analysis

Print stat summaries for all Pokémon and for each primary type.

In [ ]:
#overall summary
overall=pokemonAnalyzer(df)
overall.printReport()

#per-type summaries
uniqueTypes=sorted(df['primary_type'].unique()) #get sorted list of types
for t in uniqueTypes:
    analyzer=pokemonAnalyzer(df,t) #create analyzer for this type
    analyzer.printReport()
    print() #blank line between types

## Top Pokémon rankings

In [ ]:
#top 5 by total stats
print('Top 5 Pokemon by Base Stat Total:')
topTotal=pokemonAnalyzer(df).topN('total',5) #get top 5
print(topTotal.to_string(index=False))

print() #separator

#top 5 by speed
print('Top 5 Fastest Pokemon:')
topSpeed=pokemonAnalyzer(df).topN('speed',5) #get top 5
print(topSpeed.to_string(index=False))

## Visualization

Create plots to compare stats across Pokémon types.

In [ ]:
##
## Plot 1: average base stat total by primary type
## Plot 2: distribution of total stats
##
fig,axes=plt.subplots(1,2,figsize=(14,5))

#bar chart of average total by type
ax=axes[0]
typeAvg=df.groupby('primary_type')['total'].mean().sort_values(ascending=False) #compute averages
typeAvg.plot(kind='bar',ax=ax,color='steelblue',edgecolor='black',linewidth=0.5) #make bar plot
ax.set_title('Avg Base Stat Total by Type');ax.set_ylabel('Total Stats');ax.set_xlabel('');ax.grid(True,alpha=0.3,axis='y') #labels
ax.tick_params(axis='x',rotation=45) #rotate labels

#histogram of total stats
ax=axes[1]
ax.hist(df['total'],bins=15,color='coral',edgecolor='black',linewidth=0.5,alpha=0.7) #histogram
ax.axvline(df['total'].mean(),color='red',linestyle='--',linewidth=2,label=f'Mean={df["total"].mean():.0f}') #mean line
ax.set_title('Distribution of Base Stat Totals');ax.set_xlabel('Total Stats');ax.set_ylabel('Count');ax.legend();ax.grid(True,alpha=0.3) #labels

plt.tight_layout();plt.show() #render

## Radar chart

Compare the stat profiles of the three starter Pokémon
(Bulbasaur, Charmander, Squirtle) using a radar chart.

In [ ]:
##
## Build a radar chart for the 3 starter pokemon.
## Each spoke is one of the 6 base stats.
##
starters=['bulbasaur','charmander','squirtle'] #the 3 starters
starterColors=['#4CAF50','#FF5722','#2196F3'] #green, red, blue
angles=np.linspace(0,2*np.pi,len(statNames),endpoint=False).tolist() #evenly spaced angles
angles+=angles[:1] #close the polygon

fig,ax=plt.subplots(figsize=(7,7),subplot_kw=dict(polar=True)) #polar axes
for name,color in zip(starters,starterColors):
    row=df[df['name']==name] #find this pokemon
    if len(row)==0: #skip if not found
        continue
    vals=row[statNames].values.flatten().tolist() #get stat values
    vals+=vals[:1] #close polygon
    ax.plot(angles,vals,linewidth=2,label=name.capitalize(),color=color) #outline
    ax.fill(angles,vals,alpha=0.15,color=color) #fill

ax.set_xticks(angles[:-1]);ax.set_xticklabels(statNames,fontsize=10) #label spokes
ax.set_title('Starter Pokemon Stat Comparison',y=1.08,fontsize=14) #title
ax.legend(loc='upper right',bbox_to_anchor=(1.3,1.1)) #legend
plt.tight_layout();plt.show() #render

## Save results

Export the processed data to a CSV file.

In [ ]:
exportCols=['name','id','primary_type','total','hp','attack','defense','special-attack','special-defense','speed','height_m','weight_kg'] #columns to export
df[exportCols].to_csv('pokemon_stats.csv',index=False) #write CSV
print(f'Saved {len(df)} pokemon to pokemon_stats.csv') #confirm
print(f'Columns: {exportCols}') #show columns